# Point 4: protocol × model factorial at 15 m/s (RF-Stat, LSTM, BERT) and episode-overlap audits

Runs the 2×2 design (cells A/B/C/D) for each model at 15 m/s, plus the Point 5 audits: E5.3 (temporal-bucket gate), E5.4 (blocked temporal split) and E5.5 (near-duplicate audit, both regimes).

25 m/s runs are in `02_protocol_vs_model_25ms.ipynb`; the rank-flip table below reads both regimes (missing files are reported). Results are cached as one JSON per (model, speed, cell, seed) in `Detector_Results/factorial`, so a rerun resumes from the cache.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd

# Adjust to your Drive layout
P15 = '/content/drive/MyDrive/Dataset RF Jamming/Dataset_2.csv'
P25 = '/content/drive/MyDrive/Dataset RF Jamming/Dataset_1.csv'

def load_and_filter(path, target_speed):
    df = pd.read_csv(path)
    df['label'] = df['Scenario'].map({1: 0, 2: 1, 3: 2})
    df = df.sort_values(['Speed', 'Scenario', 'Time']).reset_index(drop=True)
    df['pseudo_run_id'] = (
        df['Speed'].round(2).astype(str) + '_' +
        df['Scenario'].astype(str) + '_' +
        (df['Time'] // 60).astype(int).astype(str)
    )
    speed_range = {15: (13, 17), 25: (23, 27)}
    lo, hi = speed_range[target_speed]
    return df[(df['Speed'] >= lo) & (df['Speed'] <= hi)].copy()

dfs15 = load_and_filter(P15, 15)
dfs25 = load_and_filter(P25, 25)

print('dfs15:', len(dfs15), 'baris,', dfs15['pseudo_run_id'].nunique(), 'episode')
print('dfs25:', len(dfs25), 'baris,', dfs25['pseudo_run_id'].nunique(), 'episode')

In [ ]:
import shutil, sys

shutil.copy('/content/drive/MyDrive/Detector_Results/pcmef_fixed.py',
            '/content/pcmef_fixed.py')
if '/content' not in sys.path:
    sys.path.insert(0, '/content')

import pcmef_fixed as P
print("OK, P siap dipakai")

In [ ]:
dfs15, feasible_15, n_dropped_15 = P.filter_feasible_episodes(dfs15, min_duration_sec=15)
dfs25, feasible_25, n_dropped_25 = P.filter_feasible_episodes(dfs25, min_duration_sec=15)
print(f"15m/s dibuang: {n_dropped_15} episode | 25m/s dibuang: {n_dropped_25} episode")

In [ ]:
shutil.copy('/content/drive/MyDrive/Detector_Results/pcmef_v2_infra.py', '/content/pcmef_v2_infra.py')
import pcmef_v2_infra as v2

print("=== Sanity check @ 15 m/s (setelah filter) ===")
v2.run_sanity_suite(P, dfs15, seeds=(42,))

print("\n=== Sanity check @ 25 m/s (setelah filter) ===")
v2.run_sanity_suite(P, dfs25, seeds=(42,))

In [ ]:
import os, json
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

def audit_nn_distance_excl_same_episode(pf, tr_df, te_df, window_sec=15, step_sec=1,
                                          seed=42, k_search=50):
    """
    E5.5: train->train distances recomputed EXCLUDING neighbours from the same
    episode (not only the point itself). Same pattern as the JamShield audit
    (E5.11), reused here for the Kosmanos data.
    """
    Xtr_l, ytr, ktr = pf.create_windows_keyed(tr_df, window_sec, step_sec)
    Xte_l, yte, kte = pf.create_windows_keyed(te_df, window_sec, step_sec)
    if not Xtr_l or not Xte_l:
        return None

    Xtr = pf.extract_tabular_features(Xtr_l)
    Xte = pf.extract_tabular_features(Xte_l)
    ep_tr = np.array([k[0] for k in ktr])

    sc = StandardScaler().fit(Xtr)
    Xtr_s = sc.transform(Xtr)
    Xte_s = sc.transform(Xte)

    nn_te = NearestNeighbors(n_neighbors=1).fit(Xtr_s)
    d_te_tr, _ = nn_te.kneighbors(Xte_s)
    d_te_tr = d_te_tr.ravel()

    k_eff = min(k_search, len(Xtr_s))
    nn_tr = NearestNeighbors(n_neighbors=k_eff).fit(Xtr_s)
    dists, idxs = nn_tr.kneighbors(Xtr_s)

    d_tr_tr_excl = np.full(len(Xtr_s), np.nan)
    n_gagal = 0
    for i in range(len(Xtr_s)):
        for j in range(1, k_eff):
            cand_idx = idxs[i, j]
            if ep_tr[cand_idx] != ep_tr[i]:
                d_tr_tr_excl[i] = dists[i, j]
                break
        else:
            n_gagal += 1

    if n_gagal > 0:
        print(f'  [peringatan] {n_gagal} window train gagal dapat tetangga beda-episode (naikkan k_search)')
    d_tr_tr_excl = d_tr_tr_excl[~np.isnan(d_tr_tr_excl)]
    if len(d_tr_tr_excl) == 0:
        return None

    p05_train = float(np.percentile(d_tr_tr_excl, 5))
    pct_test_closer = float(100.0 * np.mean(d_te_tr < p05_train))

    return dict(
        n_train_windows=len(Xtr), n_test_windows=len(Xte), n_gagal_exclude=int(n_gagal),
        d_test_to_train_mean=float(d_te_tr.mean()),
        d_train_to_train_excl_mean=float(d_tr_tr_excl.mean()),
        p05_train_to_train_excl=p05_train,
        pct_test_closer_than_p05_train=pct_test_closer,
    )

OUT_E55 = '/content/drive/MyDrive/Detector_Results/audit_e55'
os.makedirs(OUT_E55, exist_ok=True)

for speed, df_speed in [(15, dfs15), (25, dfs25)]:
    print(f"\n{'='*60}\n=== E5.5: Audit near-duplicate Kosmanos @ {speed} m/s ===\n{'='*60}")
    results = []
    for seed in P.SEEDS:
        fp = os.path.join(OUT_E55, f'E55_{speed}ms_seed{seed}.json')
        if os.path.exists(fp):
            results.append(json.load(open(fp))); continue
        tr_df, va_df, te_df = P.split_by_episode(df_speed, seed=seed)
        r = audit_nn_distance_excl_same_episode(P, tr_df, te_df, seed=seed)
        if r is None:
            print(f'  [seed {seed}] dilewati (data tidak cukup)'); continue
        r['seed'] = int(seed)
        json.dump(r, open(fp, 'w'), indent=2)
        results.append(r)
        print(f'  [seed {seed:4d}] pct_test_closer_than_p05 = {r["pct_test_closer_than_p05_train"]:.2f}%  '
              f'(p05_excl={r["p05_train_to_train_excl"]:.4f})')

    pct_vals = np.array([r['pct_test_closer_than_p05_train'] for r in results])
    print(f"\n=== RINGKASAN E5.5 @ {speed} m/s (n={len(pct_vals)} seed) ===")
    print(f"  pct_test_closer_than_p05_train: mean={pct_vals.mean():.2f}%  SD={pct_vals.std(ddof=1):.2f}%  "
          f"min={pct_vals.min():.2f}%  max={pct_vals.max():.2f}%")

## RF-Stat, 15 m/s

In [ ]:
import os

OUT = '/content/drive/MyDrive/Detector_Results/factorial'

# Set True to delete the cached 15 m/s RF-Stat results and recompute everything.
CLEAR_RF_CACHE = False
if CLEAR_RF_CACHE:
    for f in os.listdir(OUT):
        if f.startswith('RF_Stat_15'):
            os.remove(os.path.join(OUT, f))

res15 = P.run_factorial(dfs15, 15, 'RF_Stat', OUT, cells=P.RF_STAT_CELLS)
P.print_factorial(P.decompose_factorial(res15), 'RF-Stat @ 15 m/s')

## LSTM, 15 m/s

In [ ]:
OUT = '/content/drive/MyDrive/Detector_Results/factorial'

res_lstm15 = P.run_factorial(dfs15, 15, 'LSTM', OUT)   # cells default to all of A/B/C/D
P.print_factorial(P.decompose_factorial(res_lstm15), 'LSTM @ 15 m/s')

In [ ]:
import glob, json, collections
rows = []
for fp in sorted(glob.glob(f'{OUT}/LSTM_15ms_*_seed*.json')):
    r = json.load(open(fp))
    cnt = r.get('test_window_count_per_class', {})
    cnt = {int(k): v for k, v in cnt.items()}
    miss = [c for c in range(3) if cnt.get(c, 0) == 0]
    rows.append((fp.split('/')[-1], cnt, miss))

nol = [x for x in rows if x[2]]
print(f'{len(rows)} file LSTM 15 m/s, {len(nol)} dengan kelas support nol\n')
for name, cnt, miss in nol:
    print(name, cnt, '-> kelas kosong:', miss)

# ringkas per sel
per = collections.Counter()
for name, cnt, miss in nol:
    per[(name.split('_')[2], tuple(miss))] += 1
print('\nRingkasan (sel, kelas kosong): jumlah seed')
for k, v in sorted(per.items()): print(k, v)

## BERT, 15 m/s

In [ ]:
try:
    import torch, transformers
    print('torch:', torch.__version__, '| transformers:', transformers.__version__)
except ImportError:
    print('belum ada, install dulu')

In [ ]:
import os

OUT = '/content/drive/MyDrive/Detector_Results/factorial'

# Set True to delete the cached 15 m/s BERT results and retrain everything (~80 runs).
CLEAR_BERT_CACHE = False
if CLEAR_BERT_CACHE:
    for f in os.listdir(OUT):
        if f.startswith('BERT_15ms'):
            os.remove(os.path.join(OUT, f))

res_bert15 = P.run_factorial(dfs15, 15, 'BERT', OUT,
                             cells=P.BERT_CELLS, seeds=P.BERT_SEEDS_DEFAULT)
P.print_factorial(P.decompose_factorial(res_bert15), 'BERT @ 15 m/s')

## Rank stability across seeds (both regimes)

In [ ]:
import os, json
import numpy as np
import pandas as pd

OUT = '/content/drive/MyDrive/Detector_Results/factorial'
MODELS = ['RF_Stat', 'LSTM', 'BERT']
SPEEDS = [15, 25]
CELLS = ['A', 'B', 'C', 'D']

# --- kumpulkan semua akurasi per (model, speed, sel, seed) dari JSON yang sudah ada ---
rows = []
missing = []
for model in MODELS:
    for speed in SPEEDS:
        for cell in CELLS:
            for seed in P.SEEDS:
                fp = os.path.join(OUT, f'{model}_{speed}ms_{cell}_seed{seed}.json')
                if not os.path.exists(fp):
                    missing.append(fp); continue
                r = json.load(open(fp))
                rows.append(dict(model=model, speed=speed, cell=cell, seed=seed,
                                  accuracy=r['accuracy']))
df_all = pd.DataFrame(rows)
print(f"Total baris terkumpul: {len(df_all)}  |  file hilang: {len(missing)}")
if missing:
    print("Contoh file hilang (maks 5):", missing[:5])

# ======================================================================
# 1. Rentang min-max & persentil 5-95 per model per regime (speed, cell)
# ======================================================================
print("\n=== Rentang akurasi per model per regime (speed, sel) ===")
summary_rows = []
for (speed, cell), g in df_all.groupby(['speed', 'cell']):
    for model in MODELS:
        sub = g[g['model'] == model]['accuracy']
        if len(sub) == 0:
            continue
        summary_rows.append(dict(
            speed=speed, cell=cell, model=model, n_seed=len(sub),
            acc_mean=sub.mean(), acc_min=sub.min(), acc_max=sub.max(),
            acc_p05=np.percentile(sub, 5), acc_p95=np.percentile(sub, 95),
            acc_range=sub.max() - sub.min(),
        ))
df_summary = pd.DataFrame(summary_rows).sort_values(['speed', 'cell', 'model'])
pd.set_option('display.float_format', lambda x: f'{x*100:.2f}%' if 0 <= x <= 1 else str(x))
print(df_summary.to_string(index=False))

# ======================================================================
# 2. Rank-flip rate per pasangan model, per regime
#    Definisi: per seed, catat model mana yang akurasinya lebih tinggi.
#    rank_flip_rate = proporsi seed yang HASILNYA BEDA dari keputusan mayoritas
#    (mis. kalau M1 menang di 15/20 seed, flip_rate = 5/20 = 25%).
#    Semakin tinggi nilainya -> semakin tidak stabil urutan model tsb,
#    semakin tidak bisa dipercaya perbandingan satu-kali ala literatur.
# ======================================================================
from itertools import combinations

print("\n=== Rank-flip rate per pasangan model, per regime ===")
flip_rows = []
for (speed, cell), g in df_all.groupby(['speed', 'cell']):
    pivot = g.pivot(index='seed', columns='model', values='accuracy')
    pivot = pivot.dropna()
    for m1, m2 in combinations(MODELS, 2):
        if m1 not in pivot.columns or m2 not in pivot.columns:
            continue
        wins_m1 = (pivot[m1] > pivot[m2]).sum()
        wins_m2 = (pivot[m2] > pivot[m1]).sum()
        ties = (pivot[m1] == pivot[m2]).sum()
        n = len(pivot)
        flip_rate = min(wins_m1, wins_m2) / n if n > 0 else np.nan
        flip_rows.append(dict(speed=speed, cell=cell, pair=f'{m1} vs {m2}',
                               n_seed=n, wins_m1=wins_m1, wins_m2=wins_m2,
                               ties=ties, rank_flip_rate=flip_rate))
df_flip = pd.DataFrame(flip_rows).sort_values(['speed', 'cell', 'pair'])
pd.set_option('display.float_format', lambda x: f'{x*100:.1f}%' if 0 <= x <= 1 else str(x))
print(df_flip.to_string(index=False))

print("\n=== Regime dengan rank-flip rate TERTINGGI (paling tidak stabil urutannya) ===")
print(df_flip.sort_values('rank_flip_rate', ascending=False).head(10).to_string(index=False))

## E5.3 / E5.4: temporal-bucket gate and blocked temporal split

In [ ]:
import pandas as pd

P15 = '/content/drive/MyDrive/Dataset RF Jamming/Dataset_2.csv'
df_raw = pd.read_csv(P15)
df_raw['label'] = df_raw['Scenario'].map({1: 0, 2: 1, 3: 2})
df_raw = df_raw.sort_values(['Speed', 'Scenario', 'Time']).reset_index(drop=True)

SPEED_RANGE_15 = (13, 17)
df15 = df_raw[(df_raw['Speed'] >= SPEED_RANGE_15[0]) & (df_raw['Speed'] <= SPEED_RANGE_15[1])].copy()
df15['bucket60'] = (df15['Time'] // 60).astype(int)

print("=== E1.3: kelas yang muncul di tiap bucket 60-detik, 15 m/s ===")
ct = pd.crosstab(df15['bucket60'], df15['label'])
print(ct)

print("\n=== Jumlah baris per (bucket, kelas) ===")
for b in sorted(df15['bucket60'].unique()):
    classes_in_bucket = sorted(df15[df15['bucket60'] == b]['label'].unique())
    print(f"  bucket {b}: kelas yang muncul = {classes_in_bucket}  (lengkap 3 kelas: {len(classes_in_bucket) == 3})")

all_buckets_complete = all(
    len(df15[df15['bucket60'] == b]['label'].unique()) == 3
    for b in df15['bucket60'].unique()
)
print(f"\n>> GERBANG E5.3: apakah SEMUA bucket punya ketiga kelas? {all_buckets_complete}")
if all_buckets_complete:
    print("   -> E5.3 (temporal holdout antar-bucket) BISA dijalankan.")
else:
    print("   -> E5.3 TIDAK BISA dijalankan seperti rencana awal (minimal satu bucket")
    print("      tidak punya ketiga kelas, train/test lintas-bucket akan timpang/kosong kelas).")

In [ ]:
import pandas as pd
import numpy as np

P15 = '/content/drive/MyDrive/Dataset RF Jamming/Dataset_2.csv'
P25 = '/content/drive/MyDrive/Dataset RF Jamming/Dataset_1.csv'
SPEED_RANGE = {15: (13, 17), 25: (23, 27)}
WINDOW_SIZES = [1, 5, 15]
PURGE_GAP = 15.0

def load_speed(path, target_speed):
    df = pd.read_csv(path)
    df['label'] = df['Scenario'].map({1: 0, 2: 1, 3: 2})
    df = df.sort_values(['Speed', 'Scenario', 'Time']).reset_index(drop=True)
    lo, hi = SPEED_RANGE[target_speed]
    return df[(df['Speed'] >= lo) & (df['Speed'] <= hi)].copy()

def blocked_temporal_split(df_speed, train_frac=0.6, val_frac=0.2, purge_gap=PURGE_GAP):
    splits = {'train': [], 'val': [], 'test': []}
    stats = []
    for scen, g in df_speed.groupby('Scenario'):
        g = g.sort_values('Time')
        t_min, t_max = g['Time'].min(), g['Time'].max()
        total = t_max - t_min
        b1 = t_min + train_frac * total
        b2 = t_min + (train_frac + val_frac) * total
        tr = g[g['Time'] < b1]
        va = g[(g['Time'] >= b1 + purge_gap) & (g['Time'] < b2)]
        te = g[g['Time'] >= b2 + purge_gap]
        splits['train'].append(tr); splits['val'].append(va); splits['test'].append(te)
        stats.append(dict(scenario=int(scen), label=int(g['label'].iloc[0]),
                          t_min=float(t_min), t_max=float(t_max), total_duration=float(total),
                          n_rows_train=len(tr), n_rows_val=len(va), n_rows_test=len(te),
                          val_duration_after_gap=float(max(0, b2 - (b1+purge_gap))),
                          test_duration_after_gap=float(max(0, t_max - (b2+purge_gap)))))
    out = {k: pd.concat(v, ignore_index=True) if v else pd.DataFrame() for k, v in splits.items()}
    return out, pd.DataFrame(stats)

def count_windows(df_part, window_sec, step_sec=1):
    if len(df_part) == 0:
        return 0
    n = 0
    g = df_part.sort_values('Time')
    times = g['Time'].values
    t0, t_end = times[0], times[-1]
    while t0 + window_sec <= t_end:
        if ((times >= t0) & (times < t0 + window_sec)).sum() > 0:
            n += 1
        t0 += step_sec
    return n

for speed in [15, 25]:
    path = P15 if speed == 15 else P25
    df_speed = load_speed(path, speed)
    splits, stats_df = blocked_temporal_split(df_speed)

    print(f"\n{'='*70}\n=== E5.4: Blocked temporal split, {speed} m/s (purge gap={PURGE_GAP}s) ===\n{'='*70}")
    print(stats_df.to_string(index=False))

    print(f"\n-- Jumlah window per skala, per split (semua Scenario digabung) --")
    for split_name in ['train', 'val', 'test']:
        part = splits[split_name]
        print(f"  {split_name}:")
        for ws in WINDOW_SIZES:
            total_w = 0
            per_class = {}
            for scen, gg in part.groupby('Scenario'):
                n_w = count_windows(gg, ws)
                total_w += n_w
                lbl = int(gg['label'].iloc[0]) if len(gg) else None
                per_class[lbl] = per_class.get(lbl, 0) + n_w
            print(f"    window {ws:2d}s: total={total_w:4d}  per_kelas={per_class}")